# Task 04 – Segmentation and Recommendation Model
**Goal:** group customers by behaviour and build a transparent, rule-based churn-risk score.

**Caution up front:** the dataset has only **15 customers**. Results are *illustrative*, not proof. Every finding below should be re-tested on a larger dataset before real business decisions.

## 1. Load and check the data

In [1]:
import pandas as pd
df = pd.read_csv('customer_churn_sample.csv')
df['Churned'] = (df['Churn'] == 'Yes').astype(int)
print(df.shape)
print('Missing values:', df.isna().sum().sum())
print('Duplicate IDs:', df['CustomerID'].duplicated().sum())
print('Overall churn rate: {:.0%}'.format(df['Churned'].mean()))
df.head()

(15, 12)
Missing values: 0
Duplicate IDs: 0
Overall churn rate: 47%


,CustomerID,Gender,Age,TenureMonths,SubscriptionType,MonthlyCharges,TotalCharges,ContractType,SupportTickets,PaymentMethod,Churn,Churned
0,CUST-1001,Female,34,12,Basic,49.99,599.88,Month-to-Month,3,Credit Card,Yes,1
1,CUST-1002,Male,45,24,Pro,79.99,1919.76,One Year,1,Bank Transfer,No,0
2,CUST-1003,Female,29,6,Basic,49.99,299.94,Month-to-Month,5,UPI,Yes,1
3,CUST-1004,Male,52,36,Enterprise,149.99,5399.64,Two Year,0,Credit Card,No,0
4,CUST-1005,Female,41,8,Pro,79.99,639.92,Month-to-Month,4,Debit Card,Yes,1


## 2. Choose features
| Feature | Used? | Why |
|---|---|---|
| ContractType | Yes | Shows commitment level |
| TenureMonths | Yes | New customers are less settled |
| SupportTickets | Yes | Many tickets signal frustration |
| SubscriptionType, MonthlyCharges | Context only | Used to describe segments / size revenue at risk |
| CustomerID | No | Just a label |
| Gender, Age | No | Weak business logic and fairness risk |
| TotalCharges | No | Just tenure × monthly charge (duplicates information) |
| PaymentMethod | No | No clear reason it drives churn; tiny sample |

Quick look at the three chosen features against churn:

In [2]:
print(df.groupby('ContractType')['Churned'].agg(['count','mean']).round(2))
print()
print(df.groupby('SupportTickets')['Churned'].agg(['count','mean']).round(2))
print()
print(df.groupby(pd.cut(df['TenureMonths'],[0,6,12,24,100]))['Churned'].agg(['count','mean']).round(2))

                count  mean
ContractType               
Month-to-Month      7   1.0
One Year            4   0.0
Two Year            4   0.0

                count  mean
SupportTickets             
0                   2   0.0
1                   4   0.0
2                   2   0.0
3                   2   1.0
4                   2   1.0
5                   2   1.0
6                   1   1.0

              count  mean
TenureMonths             
(0, 6]            4   1.0
(6, 12]           2   1.0
(12, 24]          5   0.2
(24, 100]         4   0.0


## 3. Create customer segments
Simple, rule-based groups. Each one can be explained in a single sentence.

1. **New & Frustrated** – Month-to-Month, tenure ≤ 12 months
2. **Flexible Established** – Month-to-Month, tenure > 12 months
3. **Mid-Term Committed** – One Year contract
4. **Loyal Long-Term** – Two Year contract

In [3]:
def assign_segment(row):
    if row['ContractType'] == 'Month-to-Month':
        return 'New & Frustrated' if row['TenureMonths'] <= 12 else 'Flexible Established'
    if row['ContractType'] == 'One Year':
        return 'Mid-Term Committed'
    return 'Loyal Long-Term'

df['Segment'] = df.apply(assign_segment, axis=1)

seg = (df.groupby('Segment')
         .agg(Customers=('CustomerID','count'),
              AvgTenure=('TenureMonths','mean'),
              AvgTickets=('SupportTickets','mean'),
              AvgMonthlyCharge=('MonthlyCharges','mean'),
              ChurnRate=('Churned','mean'))
         .round(2).reset_index())
seg.to_csv('segment_table.csv', index=False)
seg

,Segment,Customers,AvgTenure,AvgTickets,AvgMonthlyCharge,ChurnRate
0,Flexible Established,1,14.00,3.0,79.99,1.0
1,Loyal Long-Term,4,38.50,0.5,149.99,0.0
2,Mid-Term Committed,4,19.75,1.5,64.99,0.0
3,New & Frustrated,6,5.83,4.5,54.99,1.0


## 4. Churn-risk score (transparent rules)
Add points, then label the total.

| Rule | Points |
|---|---|
| Month-to-Month contract | +3 |
| One Year contract | +1 |
| Two Year contract | 0 |
| Tenure ≤ 6 months | +2 |
| Tenure 7–12 months | +1 |
| Tenure > 12 months | 0 |
| 4+ support tickets | +2 |
| 2–3 support tickets | +1 |
| 0–1 support tickets | 0 |

**Risk band:** 0–2 = Low, 3–4 = Medium, 5+ = High. (Maximum score = 7.)

In [4]:
def churn_risk_score(contract, tenure, tickets):
    # Edge-case handling: missing or invalid values never crash the score
    if pd.isna(contract) or pd.isna(tenure) or pd.isna(tickets):
        return None
    if tenure < 0 or tickets < 0:
        return None
    pts = {'Month-to-Month': 3, 'One Year': 1, 'Two Year': 0}.get(contract)
    if pts is None:
        return None            # unknown contract type -> flag for review
    pts += 2 if tenure <= 6 else (1 if tenure <= 12 else 0)
    pts += 2 if tickets >= 4 else (1 if tickets >= 2 else 0)
    return pts

def risk_band(score):
    if score is None: return 'Needs review'
    return 'Low' if score <= 2 else ('Medium' if score <= 4 else 'High')

df['RiskScore'] = df.apply(lambda r: churn_risk_score(r['ContractType'], r['TenureMonths'], r['SupportTickets']), axis=1)
df['RiskBand'] = df['RiskScore'].apply(risk_band)
df[['CustomerID','Segment','RiskScore','RiskBand','Churn']].sort_values('RiskScore', ascending=False)

,CustomerID,Segment,RiskScore,RiskBand,Churn
2,CUST-1003,New & Frustrated,7,High,Yes
6,CUST-1007,New & Frustrated,7,High,Yes
9,CUST-1010,New & Frustrated,7,High,Yes
13,CUST-1014,New & Frustrated,7,High,Yes
4,CUST-1005,New & Frustrated,6,High,Yes
0,CUST-1001,New & Frustrated,5,High,Yes
11,CUST-1012,Flexible Established,4,Medium,Yes
5,CUST-1006,Mid-Term Committed,2,Low,No
8,CUST-1009,Mid-Term Committed,2,Low,No
12,CUST-1013,Mid-Term Committed,1,Low,No


## 5. Validate: does the score actually work?

In [5]:
val = (df.groupby('RiskBand')
         .agg(Customers=('CustomerID','count'), ChurnRate=('Churned','mean'))
         .reindex(['Low','Medium','High']).round(2))
val

,Customers,ChurnRate
RiskBand,,
Low,8,0.0
Medium,1,1.0
High,6,1.0


In [6]:
# Reproducibility check: running the score again gives identical results
again = df.apply(lambda r: churn_risk_score(r['ContractType'], r['TenureMonths'], r['SupportTickets']), axis=1)
print('Reproducible:', (again == df['RiskScore']).all())

# Is the order sensible? Higher band should not churn less than a lower band
rates = val['ChurnRate'].dropna().tolist()
print('Churn rises with risk band:', rates == sorted(rates))

Reproducible: True
Churn rises with risk band: True


### Edge cases

In [7]:
tests = [
    ('Brand-new customer, month-to-month, 0 tickets', 'Month-to-Month', 0, 0),
    ('Very long tenure, month-to-month, many tickets', 'Month-to-Month', 60, 8),
    ('Two-year contract but 10 tickets', 'Two Year', 24, 10),
    ('Missing contract type', None, 12, 1),
    ('Unknown contract label', 'Weekly', 12, 1),
    ('Negative tenure (bad data)', 'One Year', -3, 1),
]
pd.DataFrame([(n, churn_risk_score(c,t,k), risk_band(churn_risk_score(c,t,k))) for n,c,t,k in tests],
             columns=['Case','Score','Band'])

,Case,Score,Band
0,"Brand-new customer, month-to-month, 0 tickets",5.0,High
1,"Very long tenure, month-to-month, many tickets",5.0,High
2,Two-year contract but 10 tickets,2.0,Low
3,Missing contract type,NaN,Needs review
4,Unknown contract label,NaN,Needs review
5,Negative tenure (bad data),NaN,Needs review


**What the edge cases show**
- A brand-new month-to-month customer with zero tickets scores 5 (High) from contract and tenure alone. That is intentional.
- Bad or missing data returns **Needs review** instead of a wrong score.
- A two-year customer with 10 tickets scores only 2 (Low). **This is a weakness:** the score under-reacts to a very unhappy loyal customer, so ticket spikes should be monitored separately.

## 6. Revenue at risk

In [8]:
at_risk = df[df['RiskBand'].isin(['High','Medium'])]
print('Customers in Medium/High risk:', len(at_risk))
print('Monthly revenue in those bands: ${:,.2f}'.format(at_risk['MonthlyCharges'].sum()))
print('Share of total monthly revenue: {:.0%}'.format(at_risk['MonthlyCharges'].sum()/df['MonthlyCharges'].sum()))
df.to_csv('customers_scored.csv', index=False)

Customers in Medium/High risk: 7
Monthly revenue in those bands: $409.93
Share of total monthly revenue: 32%


## 7. Recommendation memo (short)
See `recommendation_memo.md` in the same folder.